# BusMatch SP
Recomendação de linha de ônibus **direta** (sem baldeação) usando a API Olho Vivo (SPTrans).

**Integrantes:** Larissa, Luiza, Moises e Thiago

**Como funciona (resumo do algoritmo):**
1. Valida as coordenadas de origem e destino.
2. Autentica na API e monta uma base local de paradas por linha/sentido.
3. Calcula (Haversine) as paradas a até `RAIO_METROS` da origem e do destino.
4. Mantém só as linhas com embarque **antes** do desembarque no mesmo sentido.
5. Escolhe a opção com menor caminhada total que tenha previsão de chegada.
6. Mostra resumo, distâncias, previsão e mapa interativo.

**Token:** guardado como segredo `SPTRANS_TOKEN` (Colab) ou variável de ambiente

In [1]:
import json
import os
import random
import time
from pathlib import Path
from math import radians, sin, cos, asin, sqrt
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
import folium
from tqdm.auto import tqdm
from IPython.display import display

# Token: segredo do Colab
try:
    from google.colab import userdata
    TOKEN = userdata.get("SPTRANS_TOKEN")
except Exception:
    TOKEN = os.environ.get("SPTRANS_TOKEN")

if not TOKEN:
    raise RuntimeError("Token não encontrado. Crie o segredo SPTRANS_TOKEN (Colab) "
                       "ou a variável de ambiente SPTRANS_TOKEN.")

BASE_URL = "http://api.olhovivo.sptrans.com.br/v2.1"

# Raio de proximidade (entre 1 e 1000 m). ~500 m = cerca de 6 a 7 min de caminhada.
RAIO_METROS = 500

# Termos para descobrir linhas: a API não busca por coordenada,
# então varremos os dígitos 0 a 9 (a busca aceita nome/número parcial).
TERMOS_BUSCA = [str(n) for n in range(10)]

ARQUIVO_BASE = Path("base_paradas.csv")
BASE = None   # será preenchida na célula que carrega ou constrói a base

print("Token carregado:", bool(TOKEN))

Token carregado: True


In [2]:
# Caixa que acompanha os extremos do município de São Paulo (norte, sul, oeste, leste).
# É uma aproximação retangular: não exclui trechos de cidades vizinhas na divisa.
LAT_MIN, LAT_MAX = -24.01, -23.35
LON_MIN, LON_MAX = -46.83, -46.36

def validar_coordenada(lat, lon, nome="coordenada"):
    """Valida tipo e limites. Retorna (lat, lon) como float."""
    try:
        lat, lon = float(lat), float(lon)
    except (TypeError, ValueError):
        raise ValueError(f"{nome}: latitude e longitude devem ser numéricas.")
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        raise ValueError(f"{nome}: valores fora dos limites geográficos.")
    if not (LAT_MIN <= lat <= LAT_MAX and LON_MIN <= lon <= LON_MAX):
        raise ValueError(f"{nome}: fora do município de São Paulo.")
    return lat, lon

def gerar_coordenada_aleatoria():
    return (round(random.uniform(LAT_MIN, LAT_MAX), 6),
            round(random.uniform(LON_MIN, LON_MAX), 6))

def haversine(lat1, lon1, lat2, lon2):
    """Distância em metros entre dois pontos (fórmula de Haversine)."""
    R = 6371000
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * R * asin(sqrt(a))

print(validar_coordenada(-23.5614, -46.6559))
print(round(haversine(-23.5505, -46.6333, -23.5614, -46.6559)), "m (esperado: 3 a 4 mil)")

(-23.5614, -46.6559)
2603 m (esperado: 3 a 4 mil)


In [3]:
sessao = requests.Session()   # mantém o cookie de autenticação

def autenticar():
    """Autentica na API. A sessão guarda o cookie para as próximas chamadas."""
    try:
        r = sessao.post(f"{BASE_URL}/Login/Autenticar",
                        params={"token": TOKEN}, timeout=15)
        r.raise_for_status()
    except requests.RequestException:
        # 'from None' evita mostrar a URL (que contém o token) no erro
        raise ConnectionError("Falha de comunicação ao autenticar na API.") from None
    if r.text.strip().lower() != "true":
        raise PermissionError("Falha de autenticação: token inválido ou expirado.")

def chamar_api(endpoint, params=None, tentativas=3):
    """GET com novas tentativas e reautenticação se a sessão expirar."""
    for t in range(tentativas):
        try:
            r = sessao.get(f"{BASE_URL}{endpoint}", params=params, timeout=20)
            if r.status_code == 401:
                autenticar()
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            time.sleep(1.5 * (t + 1))
    raise ConnectionError(f"Falha ao consultar {endpoint} após {tentativas} tentativas.")

autenticar()
print("Autenticado com sucesso")

Autenticado com sucesso


In [4]:
amostra = chamar_api("/Linha/Buscar", {"termosBusca": "8000"})
print(len(amostra), "linhas")
print(amostra[:1])

paradas = chamar_api("/Parada/BuscarParadasPorLinha", {"codigoLinha": amostra[0]["cl"]})
print(len(paradas), "paradas")
print(paradas[:2])

4 linhas
[{'cl': 2506, 'lc': False, 'lt': '8000', 'sl': 1, 'tl': 1, 'tp': 'PÇA. RAMOS DE AZEVEDO', 'ts': 'TERM. LAPA'}]
14 paradas
[{'cp': 7014417, 'np': 'ANGELICA B/C', 'ed': 'AV   ANGELICA', 'py': -23.534564, 'px': -46.654302}, {'cp': 60016784, 'np': 'PARADA PALMEIRAS B/C', 'ed': 'R PADRE ANTONIO TOMAS/ AV POMPEIA', 'py': -23.525799, 'px': -46.679251}]


In [5]:
def buscar_linhas(termos):
    """Reúne linhas (cada sentido tem um código 'cl' próprio), sem duplicar."""
    linhas = {}
    for termo in tqdm(termos, desc="Buscando linhas"):
        for l in chamar_api("/Linha/Buscar", {"termosBusca": termo}) or []:
            linhas[l["cl"]] = l
    if not linhas:
        raise LookupError("Nenhuma linha retornada pela API para os termos informados.")
    return pd.DataFrame(linhas.values())

def baixar_paradas(linha):
    """Paradas de uma linha/sentido, na ordem retornada pela API (seq)."""
    dados = chamar_api("/Parada/BuscarParadasPorLinha", {"codigoLinha": linha["cl"]}) or []
    return [{
        "cl": linha["cl"],
        "letreiro": f"{linha.get('lt', '?')}-{linha.get('tl', '?')}",
        "sentido": linha.get("sl"),
        "tp": linha.get("tp"),
        "ts": linha.get("ts"),
        "seq": i,
        "cp": p.get("cp"),
        "np": p.get("np"),
        "lat": p.get("py"),
        "lon": p.get("px"),
    } for i, p in enumerate(dados)]

def construir_base(termos, max_linhas=None, workers=5, salvar=True):
    linhas_df = buscar_linhas(termos)
    print(len(linhas_df), "linhas/sentidos encontrados")
    if max_linhas:
        linhas_df = linhas_df.head(max_linhas)

    registros, falhas = [], 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futuros = [ex.submit(baixar_paradas, l) for l in linhas_df.to_dict("records")]
        for f in tqdm(as_completed(futuros), total=len(futuros), desc="Baixando paradas"):
            try:
                registros.extend(f.result())
            except Exception:
                falhas += 1

    if not registros:
        raise LookupError("Nenhuma parada foi retornada pela API.")
    base = pd.DataFrame(registros)
    base["lat"] = pd.to_numeric(base["lat"], errors="coerce")
    base["lon"] = pd.to_numeric(base["lon"], errors="coerce")
    base = base.dropna(subset=["lat", "lon"]).sort_values(["cl", "seq"]).reset_index(drop=True)

    sem_paradas = linhas_df["cl"].nunique() - base["cl"].nunique()
    print(f"Base: {len(base)} paradas de {base['cl'].nunique()} linhas/sentidos "
          f"({falhas} falhas; {sem_paradas} linhas/sentidos sem paradas na API foram ignorados)")
    if salvar:
        base.to_csv(ARQUIVO_BASE, index=False)
    return base

In [6]:
BASE = construir_base(TERMOS_BUSCA, max_linhas=40, salvar=False)
BASE.head()

Buscando linhas:   0%|          | 0/10 [00:00<?, ?it/s]

2398 linhas/sentidos encontrados


Baixando paradas:   0%|          | 0/40 [00:00<?, ?it/s]

Base: 19 paradas de 5 linhas/sentidos (0 falhas; 35 linhas/sentidos sem paradas na API foram ignorados)


,cl,letreiro,sentido,tp,ts,seq,cp,np,lat,lon
0,516,106A-10,1,ITAIM BIBI,METRÔ SANTANA,0,706334,PARADA 14 BIS C/B,-23.555608,-46.649766
1,516,106A-10,1,ITAIM BIBI,METRÔ SANTANA,1,70016562,PARADA 2 - GETÚLIO VARGAS C/B,-23.559239,-46.654314
2,516,106A-10,1,ITAIM BIBI,METRÔ SANTANA,2,440015157,PARADA 2 - GUIANAS - C/B,-23.573109,-46.665950
3,516,106A-10,1,ITAIM BIBI,METRÔ SANTANA,3,440015159,PARADA 2 - ESTADOS UNIDOS C/B,-23.570309,-46.662178
4,516,106A-10,1,ITAIM BIBI,METRÔ SANTANA,4,440015162,PARADA 2 - JOSÉ MARIA LISBOA C/B,-23.567434,-46.659457


In [7]:
if ARQUIVO_BASE.exists():
    BASE = pd.read_csv(ARQUIVO_BASE)
    print(f"Base carregada do arquivo: {len(BASE)} paradas, {BASE['cl'].nunique()} linhas/sentidos")
else:
    BASE = construir_base(TERMOS_BUSCA)

Buscando linhas:   0%|          | 0/10 [00:00<?, ?it/s]

2398 linhas/sentidos encontrados


Baixando paradas:   0%|          | 0/2398 [00:00<?, ?it/s]

Base: 4439 paradas de 563 linhas/sentidos (0 falhas; 1835 linhas/sentidos sem paradas na API foram ignorados)


In [8]:
def encontrar_candidatas(base, origem, destino, raio):
    base = base.copy()
    base["d_origem"] = [haversine(origem[0], origem[1], la, lo)
                        for la, lo in zip(base["lat"], base["lon"])]
    base["d_destino"] = [haversine(destino[0], destino[1], la, lo)
                         for la, lo in zip(base["lat"], base["lon"])]
    candidatas = []
    for cl, grupo in base.groupby("cl"):
        perto_o = grupo[grupo["d_origem"] <= raio]
        perto_d = grupo[grupo["d_destino"] <= raio]
        if perto_o.empty or perto_d.empty:
            continue
        melhor = None
        for _, o in perto_o.iterrows():
            for _, d in perto_d.iterrows():
                if o["seq"] < d["seq"]:               # embarque antes do desembarque
                    custo = o["d_origem"] + d["d_destino"]
                    if melhor is None or custo < melhor[0]:
                        melhor = (custo, o, d)
        if melhor is None:
            continue                                   # só existe no sentido contrário
        _, emb, des = melhor
        candidatas.append({
            "codigo_linha": int(cl),
            "letreiro": emb["letreiro"],
            "sentido": emb["sentido"],
            "terminal_principal": emb["tp"],
            "terminal_secundario": emb["ts"],
            "embarque": emb.to_dict(),
            "desembarque": des.to_dict(),
            "dist_embarque": float(emb["d_origem"]),
            "dist_desembarque": float(des["d_destino"]),
            "paradas": grupo.sort_values("seq"),
        })
    return candidatas

def chave_opcao(c):
    """Critério: menor caminhada total; empate = trecho com menos paradas."""
    return (c["dist_embarque"] + c["dist_desembarque"],
            c["desembarque"]["seq"] - c["embarque"]["seq"])

def ordenar_candidatas(candidatas):
    return sorted(candidatas, key=chave_opcao)

def selecionar_opcao(candidatas):
    if not candidatas:
        return None
    return min(candidatas, key=chave_opcao)

def detalhar_opcao(opcao):
    g = opcao["paradas"]
    i, j = opcao["embarque"]["seq"], opcao["desembarque"]["seq"]
    inter = g[(g["seq"] > i) & (g["seq"] < j)].to_dict("records")
    return {"embarque": opcao["embarque"], "desembarque": opcao["desembarque"],
            "intermediarias": inter}

def diagnosticar(base, origem, destino, raio):
    """Explica por que nenhuma linha foi encontrada."""
    b = base.assign(
        d_o=[haversine(origem[0], origem[1], la, lo) for la, lo in zip(base["lat"], base["lon"])],
        d_d=[haversine(destino[0], destino[1], la, lo) for la, lo in zip(base["lat"], base["lon"])])
    print(f"Linhas/sentidos na base: {b['cl'].nunique()}")
    print(f"Com parada a até {raio} m da origem:  {b[b.d_o <= raio]['cl'].nunique()}")
    print(f"Com parada a até {raio} m do destino: {b[b.d_d <= raio]['cl'].nunique()}")
    print(f"Parada mais próxima da origem:  {b.d_o.min():.0f} m")
    print(f"Parada mais próxima do destino: {b.d_d.min():.0f} m")

In [9]:
def minutos_entre(hora_a, hora_b):
    """Minutos entre dois horários 'HH:MM' (considera virada de meia-noite)."""
    try:
        ha, ma = map(int, str(hora_a).split(":"))
        hb, mb = map(int, str(hora_b).split(":"))
    except ValueError:
        return None
    return ((hb * 60 + mb) - (ha * 60 + ma)) % 1440

def _veiculo(v, hora_consulta=None):
    """Padroniza um veículo da resposta: prefixo, horário previsto, minutos até chegar, posição."""
    t = v.get("t")
    return {
        "prefixo": v.get("p"),
        "previsao": t,
        "minutos": minutos_entre(hora_consulta, t) if hora_consulta else None,
        "acessivel": v.get("a"),
        "lat": v.get("py"),
        "lon": v.get("px"),
    }

def extrair_previsoes(no, hora_consulta=None):
    """Procura recursivamente as listas 'vs' (veículos) na resposta da API."""
    achados = []
    if isinstance(no, dict):
        if isinstance(no.get("vs"), list):
            for v in no["vs"]:
                if isinstance(v, dict) and v.get("t"):
                    achados.append(_veiculo(v, hora_consulta))
        for valor in no.values():
            if isinstance(valor, (dict, list)):
                achados += extrair_previsoes(valor, hora_consulta)
    elif isinstance(no, list):
        for item in no:
            achados += extrair_previsoes(item, hora_consulta)
    return achados

def ordenar_por_chegada(previsoes):
    """O primeiro da lista é o próximo ônibus a chegar."""
    return sorted(previsoes, key=lambda v: (v["minutos"] is None,
                                            v["minutos"] if v["minutos"] is not None else 0,
                                            str(v["previsao"])))

def prever_chegada(codigo_parada, codigo_linha):
    """Previsão de uma linha em uma parada (/Previsao). Lista ordenada: [0] = próximo ônibus."""
    dados = chamar_api("/Previsao", {"codigoParada": int(codigo_parada),
                                      "codigoLinha": int(codigo_linha)})
    hora = dados.get("hr") if isinstance(dados, dict) else None
    return ordenar_por_chegada(extrair_previsoes(dados, hora))

def prever_linha(codigo_linha):
    """Previsões da linha inteira (/Previsao/Linha): {código da parada: [veículos]}.
    Serve para mostrar a previsão em outros pontos da linha (Req. 05)."""
    dados = chamar_api("/Previsao/Linha", {"codigoLinha": int(codigo_linha)})
    hora = dados.get("hr") if isinstance(dados, dict) else None
    resultado = {}
    for p in (dados.get("ps") if isinstance(dados, dict) else None) or []:
        if not isinstance(p, dict) or p.get("cp") is None:
            continue
        prev = ordenar_por_chegada(extrair_previsoes(p, hora))
        if prev:
            resultado[int(p["cp"])] = prev
    return resultado

def formatar_chegada(v):
    """Ex.: 'em 7 min (às 10:07)' ou 'chegando (às 10:00)'."""
    if v.get("minutos") is None:
        return f"às {v['previsao']}"
    if v["minutos"] == 0:
        return f"chegando (às {v['previsao']})"
    return f"em {v['minutos']} min (às {v['previsao']})"

def descrever_veiculo(v):
    acess = " ♿" if v.get("acessivel") else ""
    return f"prefixo {v['prefixo']} — {formatar_chegada(v)}{acess}"

In [10]:
def descrever_sentido(op):
    """sl=1: terminal principal -> secundário; sl=2: secundário -> principal."""
    tp, ts = op["terminal_principal"], op["terminal_secundario"]
    return f"{tp} → {ts}" if op["sentido"] == 1 else f"{ts} → {tp}"

def comprimento_trecho(emb, inter, des):
    """Distância aproximada do trecho de ônibus (soma entre paradas consecutivas)."""
    pts = [(emb["lat"], emb["lon"])] + [(p["lat"], p["lon"]) for p in inter] + [(des["lat"], des["lon"])]
    return sum(haversine(a[0], a[1], b[0], b[1]) for a, b in zip(pts, pts[1:]))

def minutos_a_pe(metros):
    """Tempo de caminhada estimado (~80 m/min, cerca de 4,8 km/h)."""
    return max(1, round(metros / 80))

def codigo_parada(p):
    try:
        return int(p["cp"])
    except (TypeError, ValueError):
        return None

def exibir_resumo(origem, destino, opcao, detalhes, previsoes, candidatas, raio=RAIO_METROS,
                  previsoes_destino=None, previsoes_paradas=None):
    previsoes = previsoes or []
    previsoes_destino = previsoes_destino or []
    previsoes_paradas = previsoes_paradas or {}
    emb, des, inter = detalhes["embarque"], detalhes["desembarque"], detalhes["intermediarias"]
    linha_reta = haversine(origem[0], origem[1], destino[0], destino[1])
    trecho = comprimento_trecho(emb, inter, des)
    a_pe_total = opcao["dist_embarque"] + opcao["dist_desembarque"]

    # ── Resumo da recomendação (Req. 04, 05 e 06) ──
    print("═" * 62)
    print("✅ RECOMENDAÇÃO")
    print(f"   Pegue a linha {opcao['letreiro']} ({descrever_sentido(opcao)})")
    print(f"   Embarque   : {emb['np']} — a {opcao['dist_embarque']:,.0f} m "
          f"(~{minutos_a_pe(opcao['dist_embarque'])} min a pé) da origem")
    if previsoes:
        print(f"   Próximo ônibus: {descrever_veiculo(previsoes[0])}")
    else:
        print("   Próximo ônibus: previsão indisponível no momento")
    print(f"   Desembarque: {des['np']} — a {opcao['dist_desembarque']:,.0f} m "
          f"(~{minutos_a_pe(opcao['dist_desembarque'])} min a pé) do destino")
    print("═" * 62)

    print("📌 REQ. 01 — COORDENADAS VALIDADAS")
    print(f"   ✔ Origem : {origem[0]:.6f}, {origem[1]:.6f}")
    print(f"   ✔ Destino: {destino[0]:.6f}, {destino[1]:.6f}")
    print(f"   ✔ Raio   : {raio:g} m")
    print("─" * 62)
    print("🔐 REQ. 02 — API OLHO VIVO")
    print(f"   ✔ Autenticado | base com {BASE['cl'].nunique()} linhas/sentidos e {len(BASE)} paradas")
    print("─" * 62)
    print(f"🔎 REQ. 03 — LINHAS CANDIDATAS: {len(candidatas)} encontrada(s) (10 melhores abaixo)")
    tabela = pd.DataFrame([{
        "Linha": c["letreiro"],
        "Sentido": c["sentido"],
        "Cam. origem (m)": round(c["dist_embarque"]),
        "Cam. destino (m)": round(c["dist_desembarque"]),
        "Total (m)": round(c["dist_embarque"] + c["dist_desembarque"]),
        "Paradas no trecho": c["desembarque"]["seq"] - c["embarque"]["seq"] + 1,
    } for c in candidatas[:10]])
    display(tabela)

    print("─" * 62)
    print("🚌 REQ. 04 — OPÇÃO SELECIONADA")
    print(f"   Linha  : {opcao['letreiro']}  (código {opcao['codigo_linha']})")
    print(f"   Sentido: {descrever_sentido(opcao)}")
    print(f"   Embarque    : {emb['np']}  (parada {emb['cp']})")
    print(f"   Desembarque : {des['np']}  (parada {des['cp']})")
    print(f"   Paradas entre embarque e desembarque: {len(inter)}")
    for k, p in enumerate(inter, 1):
        print(f"      {k:>2}. {p['np']}")

    # ── Req. 05: previsões ──
    print("─" * 62)
    print("⏱️ REQ. 05 — PREVISÃO DE CHEGADA")
    print(f"   Ponto de embarque — {emb['np']}:")
    if previsoes:
        print(f"      🚍 PRÓXIMO ÔNIBUS: {descrever_veiculo(previsoes[0])}")
        for v in previsoes[1:]:
            print(f"         depois: {descrever_veiculo(v)}")
    else:
        print("      Previsão indisponível no momento (sem veículos monitorados para esta parada/linha).")

    print(f"   Ponto de desembarque (perto do destino) — {des['np']}:")
    if previsoes_destino:
        por_prefixo = {v["prefixo"]: v for v in previsoes_destino}
        for v in previsoes_destino:
            print(f"      {descrever_veiculo(v)}")
        casados = [(v, por_prefixo[v["prefixo"]]) for v in previsoes if v["prefixo"] in por_prefixo]
        for v, vd in casados:
            dur = minutos_entre(v["previsao"], vd["previsao"])
            extra = f" (~{dur} min de viagem)" if dur is not None else ""
            print(f"   ➜ Prefixo {v['prefixo']}: embarca às {v['previsao']} e chega ao desembarque às {vd['previsao']}{extra}")
        if not casados:
            print("   (Não dá para confirmar qual ônibus do embarque aparece aqui; "
                  "os horários são dos próximos ônibus da linha neste ponto.)")
    else:
        print("      Previsão indisponível no momento para este ponto.")

    print("   Outros pontos da linha, no trecho da viagem:")
    linhas = []
    for p in [emb] + inter + [des]:
        prev = previsoes_paradas.get(codigo_parada(p))
        if prev:
            linhas.append({"Parada": p["np"], "Próximo ônibus": prev[0]["previsao"],
                           "Em (min)": prev[0]["minutos"], "Prefixo": prev[0]["prefixo"]})
    if linhas:
        display(pd.DataFrame(linhas))
    else:
        print("      Previsão para os outros pontos indisponível no momento.")

    # ── Req. 06: distâncias ──
    print("─" * 62)
    print("📏 REQ. 06 — DISTÂNCIAS (em linha reta, Haversine)")
    print(f"   Origem → destino            : {linha_reta:,.0f} m")
    print(f"   Origem → embarque (a pé)    : {opcao['dist_embarque']:,.0f} m")
    print(f"   Trecho de ônibus (aprox.)   : {trecho:,.0f} m")
    print(f"   Desembarque → destino (a pé): {opcao['dist_desembarque']:,.0f} m")
    print(f"   Caminhada total             : {a_pe_total:,.0f} m (~{minutos_a_pe(a_pe_total)} min)")
    print("─" * 62)
    print("🗺️ REQ. 06 — MAPA INTERATIVO a seguir")

In [11]:
def recomendar_viagem(origem, destino, raio=RAIO_METROS, max_tentativas_previsao=5):
    try:
        origem = validar_coordenada(*origem, nome="Origem")
        destino = validar_coordenada(*destino, nome="Destino")
        if haversine(origem[0], origem[1], destino[0], destino[1]) < 1:
            raise ValueError("Origem e destino são o mesmo ponto.")
        try:
            raio = float(raio)
        except (TypeError, ValueError):
            raise ValueError("O raio deve ser um número em metros.")
        if not (1 <= raio <= 1000):
            raise ValueError("O raio deve estar entre 1 e 1000 metros.")
        if BASE is None or len(BASE) == 0:
            raise LookupError("Base de paradas vazia. Execute a célula que constrói a base.")

        autenticar()
        candidatas = encontrar_candidatas(BASE, origem, destino, raio)
        if not candidatas:
            print("Nenhuma linha direta atende aos critérios. Diagnóstico:")
            diagnosticar(BASE, origem, destino, raio)
            print("Tente aumentar o raio ou ampliar os termos de busca.")
            return None

        ordenadas = ordenar_candidatas(candidatas)

        # Testa as melhores candidatas até achar uma com previsão de chegada ao embarque
        opcao, detalhes, previsoes = None, None, []
        for cand in ordenadas[:max_tentativas_previsao]:
            det = detalhar_opcao(cand)
            try:
                prev = prever_chegada(det["embarque"]["cp"], cand["codigo_linha"])
            except (ConnectionError, ValueError):
                prev = []              # a previsão é opcional; segue sem ela
            if prev:
                opcao, detalhes, previsoes = cand, det, prev
                break

        if opcao is None:              # nenhuma teve previsão: usa a melhor opção mesmo assim
            opcao = ordenadas[0]
            detalhes = detalhar_opcao(opcao)
            print("Aviso: sem previsão de chegada para as melhores opções; "
                  "mostrando a de menor caminhada.")

        # Previsões da linha inteira (outros pontos) e do ponto de desembarque
        try:
            previsoes_paradas = prever_linha(opcao["codigo_linha"])
        except (ConnectionError, ValueError):
            previsoes_paradas = {}

        try:
            previsoes_destino = prever_chegada(detalhes["desembarque"]["cp"], opcao["codigo_linha"])
        except (ConnectionError, ValueError):
            previsoes_destino = []
        if not previsoes_destino:      # plano B: usa a previsão da linha inteira
            previsoes_destino = previsoes_paradas.get(codigo_parada(detalhes["desembarque"]), [])
        if not previsoes:              # plano B também para o embarque
            previsoes = previsoes_paradas.get(codigo_parada(detalhes["embarque"]), [])

        exibir_resumo(origem, destino, opcao, detalhes, previsoes, ordenadas, raio,
                      previsoes_destino, previsoes_paradas)
        return montar_mapa(origem, destino, detalhes, opcao, previsoes,
                           previsoes_destino, previsoes_paradas)

    except (ValueError, PermissionError, ConnectionError, LookupError) as erro:
        print(f"Erro: {erro}")
        return None
    except Exception as erro:
        print(f"Erro inesperado ({type(erro).__name__}): {erro}")
        return None

In [12]:
import html as _html

def _popup(titulo, nome, cp, prevs=None):
    texto = f"<b>{_html.escape(titulo)}</b><br>{_html.escape(str(nome))}<br>Parada {cp}"
    for v in (prevs or [])[:3]:
        texto += f"<br>🚍 {v['prefixo']}: {formatar_chegada(v)}"
    return folium.Popup(texto, max_width=260)

def montar_mapa(origem, destino, detalhes, opcao=None, previsoes=None,
                previsoes_destino=None, previsoes_paradas=None):
    previsoes = previsoes or []
    previsoes_destino = previsoes_destino or []
    previsoes_paradas = previsoes_paradas or {}
    emb, des = detalhes["embarque"], detalhes["desembarque"]
    inter = detalhes["intermediarias"]

    ponto_emb = [float(emb["lat"]), float(emb["lon"])]
    ponto_des = [float(des["lat"]), float(des["lon"])]
    trajeto = [ponto_emb] + [[float(p["lat"]), float(p["lon"])] for p in inter] + [ponto_des]

    centro = [(origem[0] + destino[0]) / 2, (origem[1] + destino[1]) / 2]
    m = folium.Map(
        location=centro, zoom_start=12,
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles © Esri — Source: Esri, HERE, Garmin, OpenStreetMap contributors",
    )

    g_pontos = folium.FeatureGroup(name="Origem, destino, embarque e desembarque").add_to(m)
    g_trecho = folium.FeatureGroup(name="Paradas do trecho").add_to(m)
    g_linha = folium.FeatureGroup(name="Outras paradas da linha", show=False).add_to(m)
    g_onibus = folium.FeatureGroup(name="Ônibus recomendado").add_to(m)
    g_trajeto = folium.FeatureGroup(name="Trajeto e caminhadas").add_to(m)

    # origem e destino
    folium.Marker(list(origem), tooltip="Origem",
                  popup=f"<b>Origem</b><br>{origem[0]:.6f}, {origem[1]:.6f}",
                  icon=folium.Icon(color="green", icon="play")).add_to(g_pontos)
    folium.Marker(list(destino), tooltip="Destino",
                  popup=f"<b>Destino</b><br>{destino[0]:.6f}, {destino[1]:.6f}",
                  icon=folium.Icon(color="red", icon="stop")).add_to(g_pontos)

    # embarque e desembarque, com a previsão no popup
    folium.Marker(ponto_emb, tooltip="Embarque",
                  popup=_popup("Embarque", emb["np"], emb["cp"], previsoes),
                  icon=folium.Icon(color="blue", icon="arrow-up")).add_to(g_pontos)
    folium.Marker(ponto_des, tooltip="Desembarque",
                  popup=_popup("Desembarque", des["np"], des["cp"], previsoes_destino),
                  icon=folium.Icon(color="darkred", icon="arrow-down")).add_to(g_pontos)

    # paradas do trecho (com previsão quando disponível)
    for k, p in enumerate(inter, 1):
        folium.CircleMarker([float(p["lat"]), float(p["lon"])], radius=5,
                            color="#1f4e9c", fill=True, fill_opacity=0.9,
                            tooltip=f"{k}. {p['np']}",
                            popup=_popup(f"Parada {k} do trecho", p["np"], p["cp"],
                                         previsoes_paradas.get(codigo_parada(p)))).add_to(g_trecho)

    # outras paradas da linha (fora do trecho), ocultas por padrão
    if opcao is not None:
        i, j = emb["seq"], des["seq"]
        for p in opcao["paradas"].to_dict("records"):
            if p["seq"] < i or p["seq"] > j:
                folium.CircleMarker([float(p["lat"]), float(p["lon"])], radius=3,
                                    color="#888", fill=True, fill_opacity=0.7,
                                    tooltip=str(p["np"])).add_to(g_linha)

    # posição apenas do ônibus recomendado (o próximo a chegar ao embarque)
    vistos = set()
    for v in previsoes[:1]:
        try:
            pos = [float(v["lat"]), float(v["lon"])]
        except (TypeError, ValueError):
            continue
        if v["prefixo"] in vistos:
            continue
        vistos.add(v["prefixo"])
        folium.Marker(pos, tooltip=f"Ônibus {v['prefixo']}",
                      popup=f"<b>Ônibus {v['prefixo']}</b><br>{formatar_chegada(v)}",
                      icon=folium.Icon(color="orange", icon="bus", prefix="fa")).add_to(g_onibus)

    # trajeto do ônibus e caminhadas (com a distância no tooltip)
    folium.PolyLine(trajeto, color="blue", weight=4, opacity=0.7,
                    tooltip="Trajeto do ônibus").add_to(g_trajeto)
    d_emb = haversine(origem[0], origem[1], ponto_emb[0], ponto_emb[1])
    d_des = haversine(ponto_des[0], ponto_des[1], destino[0], destino[1])
    folium.PolyLine([list(origem), ponto_emb], color="gray", weight=3, dash_array="6",
                    tooltip=f"Caminhada até o embarque: {d_emb:,.0f} m").add_to(g_trajeto)
    folium.PolyLine([ponto_des, list(destino)], color="gray", weight=3, dash_array="6",
                    tooltip=f"Caminhada do desembarque ao destino: {d_des:,.0f} m").add_to(g_trajeto)

    # caixa de informações (linha, sentido e próximo ônibus)
    if opcao is not None:
        prox = descrever_veiculo(previsoes[0]) if previsoes else "previsão indisponível"
        caixa = (
            '<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; background: white;'
            ' padding: 8px 12px; border: 1px solid #999; border-radius: 6px; font-size: 13px; max-width: 270px;">'
            f"<b>Linha {_html.escape(str(opcao['letreiro']))}</b><br>{_html.escape(descrever_sentido(opcao))}<br>"
            f"Embarque: {_html.escape(str(emb['np']))}<br>"
            f"Desembarque: {_html.escape(str(des['np']))}<br>"
            f"Próximo ônibus: {_html.escape(prox)}<br>"
            f"Caminhada: {d_emb + d_des:,.0f} m"
            "</div>")
        m.get_root().html.add_child(folium.Element(caixa))

    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([list(origem), list(destino), ponto_emb, ponto_des] + trajeto)
    return m

In [18]:
# ───────────── EDITE APENAS AQUI ─────────────
USAR_ALEATORIO = False              # True = sorteia origem e destino
ORIGEM  = (-23.744071, -46.70707)   # perto de HILÁRIO B/C   (latitude, longitude)
DESTINO = (-23.804705, -46.73479)   # perto de ANA CAROLINA B/C
RAIO    = 500                       # metros, de 1 a 1000
# ──────────────────────────────────────────────

def titulo(txt):
    print("\n" + "=" * 62 + f"\n{txt}\n" + "=" * 62)

# REQ. 01 a 06: coordenadas, API, candidatas, seleção, previsão, resultados e mapa
titulo("DEMONSTRAÇÃO — REQUISITOS 01 A 06")
if USAR_ALEATORIO:
    ORIGEM, DESTINO = gerar_coordenada_aleatoria(), gerar_coordenada_aleatoria()
    print("Coordenadas sorteadas:", ORIGEM, DESTINO)

mapa = recomendar_viagem(ORIGEM, DESTINO, RAIO)
if mapa is not None:
    titulo("REQ. 06 — MAPA INTERATIVO (origem, destino, embarque, desembarque e paradas)")
    display(mapa)

# REQ. 07: tratamento de erros
titulo("REQUISITO 07 — TRATAMENTO DE ERROS")
casos = [
    ("Coordenada inválida",            ("abc", 1),            DESTINO,           RAIO),
    ("Fora de São Paulo",              (-10, -46),            DESTINO,           RAIO),
    ("Raio inválido",                  ORIGEM,                DESTINO,           5000),
    ("Origem igual ao destino",        ORIGEM,                ORIGEM,            RAIO),
    ("Região sem linhas no raio",      (-23.99, -46.70),      (-23.98, -46.69),  50),
]
for nome, o, d, r in casos:
    print(f"\n▶ {nome}")
    recomendar_viagem(o, d, r)

print("\n▶ Falha de autenticação (token inválido)")
token_original = TOKEN
TOKEN = "token-invalido"
try:
    recomendar_viagem(ORIGEM, DESTINO, RAIO)
finally:
    TOKEN = token_original        # restaura o token real

# Previsão indisponível e respostas vazias já são tratadas dentro de recomendar_viagem
# (testa as próximas linhas e, se preciso, avisa que não há previsão).


DEMONSTRAÇÃO — REQUISITOS 01 A 06
══════════════════════════════════════════════════════════════
✅ RECOMENDAÇÃO
   Pegue a linha 6099-10 (TERM. GRAJAÚ → DIVISA DE EMBU-GUAÇU)
   Embarque   : HILÁRIO B/C — a 78 m (~1 min a pé) da origem
   Próximo ônibus: prefixo 66198 — em 32 min (às 20:50) ♿
   Desembarque: ANA CAROLINA B/C — a 100 m (~1 min a pé) do destino
══════════════════════════════════════════════════════════════
📌 REQ. 01 — COORDENADAS VALIDADAS
   ✔ Origem : -23.744071, -46.707070
   ✔ Destino: -23.804705, -46.734790
   ✔ Raio   : 500 m
──────────────────────────────────────────────────────────────
🔐 REQ. 02 — API OLHO VIVO
   ✔ Autenticado | base com 563 linhas/sentidos e 4439 paradas
──────────────────────────────────────────────────────────────
🔎 REQ. 03 — LINHAS CANDIDATAS: 14 encontrada(s) (10 melhores abaixo)


,Linha,Sentido,Cam. origem (m),Cam. destino (m),Total (m),Paradas no trecho
0,6099-10,2,78,100,178,9
1,695Y-21,1,78,100,178,13
2,6000-10,1,78,100,178,16
3,6091-10,1,78,100,178,16
4,6091-51,1,78,100,178,16
5,695Y-10,1,78,100,178,21
6,6091-31,1,100,100,200,9
7,6091-21,1,100,100,200,19
8,6091-10,2,408,91,499,12
9,695Y-41,2,408,91,499,12


──────────────────────────────────────────────────────────────
🚌 REQ. 04 — OPÇÃO SELECIONADA
   Linha  : 6099-10  (código 34355)
   Sentido: TERM. GRAJAÚ → DIVISA DE EMBU-GUAÇU
   Embarque    : HILÁRIO B/C  (parada 230009864)
   Desembarque : ANA CAROLINA B/C  (parada 560009175)
   Paradas entre embarque e desembarque: 7
       1. JOSÉ NICOLAU DE LIMA B/C
       2. ELEVATÓRIA SABESP B/C
       3. NIELS C. SORENSEN B/C
       4. ALFREDO REIMBERG B/C
       5. ESTRADA DO PINHAL B/C
       6. BALTAZARINI B/C
       7. ESTRADA DO PAIOL B/C
──────────────────────────────────────────────────────────────
⏱️ REQ. 05 — PREVISÃO DE CHEGADA
   Ponto de embarque — HILÁRIO B/C:
      🚍 PRÓXIMO ÔNIBUS: prefixo 66198 — em 32 min (às 20:50) ♿
         depois: prefixo 66468 — em 47 min (às 21:05) ♿
   Ponto de desembarque (perto do destino) — ANA CAROLINA B/C:
      prefixo 66468 — em 9 min (às 20:27) ♿
   ➜ Prefixo 66468: embarca às 21:05 e chega ao desembarque às 20:27 (~1402 min de viagem)
   Outros

,Parada,Próximo ônibus,Em (min),Prefixo
0,HILÁRIO B/C,20:50,32,66198
1,JOSÉ NICOLAU DE LIMA B/C,20:26,8,66198
2,ELEVATÓRIA SABESP B/C,20:24,6,66198
3,NIELS C. SORENSEN B/C,20:21,3,66198
4,ALFREDO REIMBERG B/C,20:19,1,66198
5,ESTRADA DO PINHAL B/C,20:32,14,66468
6,BALTAZARINI B/C,20:29,11,66468
7,ESTRADA DO PAIOL B/C,20:28,10,66468
8,ANA CAROLINA B/C,20:27,9,66468


──────────────────────────────────────────────────────────────
📏 REQ. 06 — DISTÂNCIAS (em linha reta, Haversine)
   Origem → destino            : 7,308 m
   Origem → embarque (a pé)    : 78 m
   Trecho de ônibus (aprox.)   : 7,236 m
   Desembarque → destino (a pé): 100 m
   Caminhada total             : 178 m (~2 min)
──────────────────────────────────────────────────────────────
🗺️ REQ. 06 — MAPA INTERATIVO a seguir

REQ. 06 — MAPA INTERATIVO (origem, destino, embarque, desembarque e paradas)



REQUISITO 07 — TRATAMENTO DE ERROS

▶ Coordenada inválida
Erro: Origem: latitude e longitude devem ser numéricas.

▶ Fora de São Paulo
Erro: Origem: fora do município de São Paulo.

▶ Raio inválido
Erro: O raio deve estar entre 1 e 1000 metros.

▶ Origem igual ao destino
Erro: Origem e destino são o mesmo ponto.

▶ Região sem linhas no raio
Nenhuma linha direta atende aos critérios. Diagnóstico:
Linhas/sentidos na base: 563
Com parada a até 50.0 m da origem:  0
Com parada a até 50.0 m do destino: 0
Parada mais próxima da origem:  18465 m
Parada mais próxima do destino: 17593 m
Tente aumentar o raio ou ampliar os termos de busca.

▶ Falha de autenticação (token inválido)
Erro: Falha de autenticação: token inválido ou expirado.


In [16]:
if mapa is not None:
    mapa.save("mapa_viagem.html")
    print("Mapa salvo. Baixe pelo painel de Arquivos (ícone de pasta).")

Mapa salvo. Baixe pelo painel de Arquivos (ícone de pasta).
